In [9]:
!pip install pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 1.2 MB/s eta 0:00:00-:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 906.7 kB/s eta 0:00:000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 3.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 5.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 509.2/509.2 kB 7.1 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [7]:
# Cellule 1: Imports et setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # Ajuste selon ta structure
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from database import Base
from db_models import User, EventType, UserEventTypeMembership
from utils import get_password_hash

# Cellule 2: Créer la DB en mémoire
engine = create_engine("sqlite:///:memory:")
Base.metadata.create_all(bind=engine)
Session = sessionmaker(bind=engine)
db = Session()

# Cellule 3: Créer les EventTypes
open_play = EventType(
    name='open_play',
    display_name='Intérieur',
    default_location='Calgary Arena',
    default_time_start='19:00',
    default_time_end='21:00',
    default_max_capacity=20,
    color='#4A90E2'
)
competitive = EventType(
    name='competitive',
    display_name='Outdoor',
    default_location='Cedarbrae Arena',
    default_time_start='17:00',
    default_time_end='19:00',
    default_max_capacity=16,
    color='#4A90E2'
)

db.add(open_play)
db.add(competitive)

db.commit()
print(f"✓ EventType créé: {open_play.id}")

# Cellule 4: Créer un User
user = User(
    email="test@example.com",
    hashed_password=get_password_hash("password123"),
    real_name="Test User",
    display_name="Test"
)
db.add(user)
db.commit()
db.refresh(user)
print(f"✓ User créé: {user.id}, {user.email}")

# Cellule 5: Vérifier en DB
users = db.query(User).all()
for u in users:
    print(f"User: {u.email}, {u.display_name} {u.id}")

✓ EventType créé: 1
✓ User créé: 1, test@example.com
User: test@example.com, Test 1


In [8]:
# Créer membership pour open_play (punch_card)
membership_open = UserEventTypeMembership(
    user_id=user.id,
    event_type_id=1,  # open_play
    membership_type='punch_card',
    total_credits_purchased=5
)
membership_competitive = UserEventTypeMembership(
    user_id=user.id,
    event_type_id=2,  # open_play
    membership_type='full_member'
)
db.add(membership_open)
db.add(membership_competitive)

In [9]:
db.commit()

# ACT - Récupérer les memberships depuis la DB
memberships = db.query(UserEventTypeMembership).filter_by(
    user_id=user.id
).all()


In [15]:
# Users
print("=== USERS ===")
users_df = pd.read_sql("SELECT * FROM users", db.bind)
display(users_df)

=== USERS ===


,id,email,display_name,real_name,hashed_password,created_at
0,1,test@example.com,Test,Test User,$2b$12$QbLSGiarSdm/1D9K03meGukv4BuiYIkRgACRwvc...,2025-12-01 04:55:38
1,2,bob@test.com,Bob,Bob Martin,$2b$12$sPfcHifeY6MpuI7CsLKRgeRXUFAkE/nuTIfMlcD...,2025-12-01 04:56:00


In [14]:
# EventTypes
print("\n=== EVENT TYPES ===")
event_types_df = pd.read_sql("SELECT * FROM event_types", db.bind)
display(event_types_df)



=== EVENT TYPES ===


,id,name,display_name,default_location,default_time_start,default_time_end,default_max_capacity,color,created_at
0,1,open_play,Intérieur,Calgary Arena,19:00,21:00,20,#4A90E2,2025-12-01 04:55:38
1,2,competitive,Outdoor,Cedarbrae Arena,17:00,19:00,16,#4A90E2,2025-12-01 04:55:38


In [16]:
# Memberships
print("\n=== MEMBERSHIPS ===")
memberships_df = pd.read_sql("SELECT * FROM user_event_type_memberships", db.bind)
display(memberships_df)


=== MEMBERSHIPS ===


,id,user_id,event_type_id,membership_type,total_credits_purchased,remaining_credits,created_at
0,1,1,1,punch_card,5.0,None,2025-12-01 04:55:47
1,2,1,2,full_member,NaN,None,2025-12-01 04:55:47


In [13]:
bob = User(
    email="bob@test.com",
    hashed_password=get_password_hash("password123"),
    real_name="Bob Martin",
    display_name="Bob"
)

db.add(bob)
db.commit()
db.refresh(bob)

In [47]:
# Bob: punch_card pour les 2 types
for event_type_id, credits in [(1, 3), (2, 8)]:
    membership = UserEventTypeMembership(
        user_id=bob.id,
        event_type_id=event_type_id,
        membership_type='punch_card',
        total_credits_purchased=credits
    )
    db.add(membership)
db.commit()

In [48]:
print("\n=== competitive_membership ===")
query = """
SELECT * FROM user_event_type_memberships 
WHERE event_type_id = 2
"""
memberships_df = pd.read_sql(query, db.bind)
display(memberships_df)


=== competitive_membership ===


,id,user_id,event_type_id,membership_type,total_credits_purchased,remaining_credits,created_at
0,2,1,2,full_member,NaN,None,2025-12-01 04:41:46
1,4,2,2,punch_card,8.0,None,2025-12-01 04:41:58


In [49]:
membership2 = UserEventTypeMembership(
        user_id=2,
        event_type_id=1,  # Même event_type!
        membership_type='full_member'
    )

# can't have double membership same evenet

In [50]:
#db.add(membership2)
#db.commit()

In [51]:
emma = User(
    email="emma@test.com",
    hashed_password=get_password_hash("password123"),
    real_name="Emma",
    display_name="Emma"
)
db.add(emma)
db.commit()
db.refresh(emma)

In [52]:
membership_open = UserEventTypeMembership(
    user_id=emma.id,
    event_type_id=1,
    membership_type='full_member',
    total_credits_purchased=None
)
db.add(membership_open)
db.commit()

In [18]:
def get_user_membership_for_event_type(db, user_id, event_type_id):
    """
    Récupère le membership d'un user pour un event_type.
    Si absent, retourne un défaut sécurisé (punch_card, 0 crédits).
    """
    membership = db.query(UserEventTypeMembership).filter_by(
        user_id=user_id,
        event_type_id=event_type_id
    ).first()
    
    if membership:
        return {
            'membership_type': membership.membership_type,
            'total_credits_purchased': membership.total_credits_purchased
        }
    else:
        # DÉFAUT DE SÉCURITÉ
        return {
            'membership_type': 'punch_card',
            'total_credits_purchased': 0
        }

In [20]:
open_membership = get_user_membership_for_event_type(db, bob.id, 1)
print(open_membership)


{'membership_type': 'punch_card', 'total_credits_purchased': 0}
